# Malang Voice Debug

Notebook ini untuk validasi list berita dan sample artikel Malang Voice sebelum dipakai di pipeline final.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'scrapers_debug':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scrapers.common import cutoff_date, parse_date
import importlib
from scrapers import malangvoice as scraper
scraper = importlib.reload(scraper)

cutoff = cutoff_date()
print('ROOT:', ROOT)
print('cutoff:', cutoff)
print('scraper file:', scraper.__file__)


ROOT: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-03-02 11:29:06.467848
scraper file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/malangvoice.py


In [2]:
MAX_PAGES = 200
TITLE_LIMIT = 90
SAMPLE_SIZE = 10  # None = semua artikel dari list_df


def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    if 'url' not in df.columns:
        df['url'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        source_category = row.get('source_category')
        category_text = '' if pd.isna(source_category) else f' | cat={source_category}'
        print(f"page={page_text} | date={date_text}{category_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))
    print('unique urls:', df['url'].nunique() if 'url' in df.columns else None)

    print('\nrows per source category:')
    if 'source_category' in df.columns:
        display(df.groupby('source_category', dropna=False).size().reset_index(name='count'))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(rows=('url', 'count'), newest=('published_dt', 'max'), oldest=('published_dt', 'min'))
        .reset_index()
        .tail(80)
    )
    return df


## Live list scrape

Malang Voice memakai pagination WordPress `/page/{n}/`, bukan load-more.

In [3]:
import importlib
from scrapers import malangvoice as scraper
scraper = importlib.reload(scraper)

list_df = scraper.scrape_list(cutoff, max_pages=MAX_PAGES, include_older=True)
print_debug_rows(list_df)
list_df = audit_list(list_df)
list_output = ROOT / 'csv' / 'malang_voice_list_debug.csv'
list_df.to_csv(list_output, index=False)
print('saved:', list_output)


Scraping Malang Voice page 1: https://malangvoice.com/kanal/malang-raya/kabupaten-malang/
HTTP GET start: https://malangvoice.com/kanal/malang-raya/kabupaten-malang/
HTTP GET ok: https://malangvoice.com/kanal/malang-raya/kabupaten-malang/ | status=200 | bytes=141736
Malang Voice list page=001 date=2026-06-28 | title=Khofifah Pastikan Pasokan Biosolar dan Pertalite di Jatim Aman
Malang Voice list page=001 date=2026-06-28 | title=Andalkan BBIB Singosari, Khofifah Optimistis Indonesia Swasembada Daging dalam Tiga Tahun
Malang Voice list page=001 date=2026-06-14 | title=Yakuza Meneges Dampingi Santri Korban Dugaan Pelecehan Seksual di Kabupaten Malang
Malang Voice list page=001 date=2026-06-11 | title=UMM Bangun Pabrik Infus di Karangploso, Perkuat Kemandirian Kesehatan Nasional
Malang Voice page 1: cards=4 new=4 total=4
Scraping Malang Voice page 2: https://malangvoice.com/kanal/malang-raya/kabupaten-malang/page/2/
HTTP GET start: https://malangvoice.com/kanal/malang-raya/kabupaten-malang

,source_category,count
0,Arema,2
1,Ekbis,13
2,Event,8
3,Gaya Hidup,1
4,Halokes,3
5,Kabupaten Malang,29



rows per month:


,month,count
0,2026-02,2
1,2026-03,16
2,2026-04,13
3,2026-05,18
4,2026-06,7



rows per page:


,page_num,rows,newest,oldest
0,1,4,2026-06-28,2026-06-11
1,2,4,2026-06-07,2026-05-31
2,3,4,2026-05-29,2026-05-26
3,4,4,2026-05-26,2026-05-25
4,5,4,2026-05-25,2026-05-19
5,6,4,2026-05-08,2026-05-02
6,7,4,2026-05-01,2026-04-24
7,8,4,2026-04-24,2026-04-22
8,9,4,2026-04-20,2026-04-17
9,10,4,2026-04-15,2026-03-29


saved: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/malang_voice_list_debug.csv


## Article sample extraction

Cell ini mengambil sample artikel dari `list_df` untuk cek kebersihan content.

In [5]:
article_rows = []
source_df = list_df if 'list_df' in globals() and len(list_df) else pd.DataFrame()

if SAMPLE_SIZE is not None:
    source_df = source_df.head(SAMPLE_SIZE)

print('article sample source rows:', len(source_df))

if len(source_df) == 0:
    print('Tidak ada row list untuk sample artikel. Jalankan cell live list dulu.')

for index, row in source_df.iterrows():
    try:
        article = scraper.extract_article(row)
        content = article.get('content') or ''

        article['content_len'] = len(content)
        article['has_literal_backslash_n'] = '\\n' in content
        article['has_actual_newline'] = '\n' in content
        article['has_double_blank_line'] = '\n\n' in content
        article['has_carriage_return'] = '\r' in content

        article_rows.append(article)

        print(
            f"article [{len(article_rows)}] len={len(content)} "
            f"| date={article.get('published_date')} "
            f"| title={short_title(article.get('title'))}"
        )
        print((content[:500] + '...') if len(content) > 500 else content)
        print('-' * 80)

    except Exception as error:
        print(f"failed [{index + 1}]: {row.get('url')} | {error}")

article_columns = [
    'published_date',
    'content_len',
    'has_literal_backslash_n',
    'has_actual_newline',
    'has_double_blank_line',
    'has_carriage_return',
    'title',
    'url',
]

article_df = pd.DataFrame(article_rows)
article_output = ROOT / 'csv' / 'malang_voice_article_debug.csv'
article_df.to_csv(article_output, index=False)

print('saved:', article_output)

if article_df.empty:
    print('No article samples extracted. Cek error di atas.')
else:
    display(article_df[[column for column in article_columns if column in article_df.columns]])

article sample source rows: 10
HTTP GET start: https://malangvoice.com/khofifah-pastikan-pasokan-biosolar-dan-pertalite-di-jatim-aman/
HTTP GET ok: https://malangvoice.com/khofifah-pastikan-pasokan-biosolar-dan-pertalite-di-jatim-aman/ | status=200 | bytes=227564
article [1] len=3439 | date=2026-06-28 | title=Khofifah Pastikan Pasokan Biosolar dan Pertalite di Jatim Aman
Gubernur Jawa Timur, Khofifah Indar Parawansa, memastikan ketersediaan bahan bakar minyak (BBM) di Jawa Timur dalam kondisi aman dan mencukupi kebutuhan masyarakat. Kepastian tersebut disampaikan usai meninjau langsung penyaluran BBM, khususnya Biosolar dan Pertalite, di kawasan Karanglo, Kabupaten Malang, Minggu (28/6). Dalam peninjauan tersebut, Khofifah menegaskan Pemerintah Provinsi Jawa Timur terus memperkuat koordinasi dengan PT Pertamina Patra Niaga dan PT PLN (Persero) guna menjaga stabi...
--------------------------------------------------------------------------------
HTTP GET start: https://malangvoice.com/a

,published_date,content_len,has_literal_backslash_n,has_actual_newline,has_double_blank_line,has_carriage_return,title,url
0,2026-06-28,3439,False,False,False,False,Khofifah Pastikan Pasokan Biosolar dan Pertali...,https://malangvoice.com/khofifah-pastikan-paso...
1,2026-06-28,3481,False,False,False,False,"Andalkan BBIB Singosari, Khofifah Optimistis I...",https://malangvoice.com/andalkan-bbib-singosar...
2,2026-06-14,2125,False,False,False,False,Yakuza Meneges Dampingi Santri Korban Dugaan P...,https://malangvoice.com/yakuza-meneges-damping...
3,2026-06-11,3398,False,False,False,False,"UMM Bangun Pabrik Infus di Karangploso, Perkua...",https://malangvoice.com/umm-bangun-pabrik-infu...
4,2026-06-07,4170,False,False,False,False,"Arema FC Buka Suara, Tegaskan Keberatan Soal P...",https://malangvoice.com/arema-fc-buka-suara-te...
5,2026-06-03,5167,False,False,False,False,PT Bentoel Group Gandeng Baskomas Jaga Kualita...,https://malangvoice.com/pt-bentoel-group-gande...
6,2026-06-01,3126,False,False,False,False,Pemuda Lintas Iman di Lawang Teguhkan Semangat...,https://malangvoice.com/pemuda-lintas-iman-di-...
7,2026-05-31,3558,False,False,False,False,"BRI Malang Catat Transaksi QRIS Rp2,05 Triliun...",https://malangvoice.com/bri-malang-catat-trans...
8,2026-05-29,2333,False,False,False,False,UMKM Desa Genengan Dapat Suntikan Bantuan CSR ...,https://malangvoice.com/umkm-desa-genengan-dap...
9,2026-05-29,3318,False,False,False,False,MPM Honda Jatim Ajak Mahasiswa Jadi Agen Perub...,https://malangvoice.com/mpm-honda-jatim-ajak-m...
